# Week 4 Debugging

In [1]:
# Read data
import io
import pandas as pd
import numpy as np

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV))

In [2]:
print(stream)

   site_id        date  water_temp_c discharge_cfs nitrate_mg_l    qc_flag
0    SB-01  2024-01-15          12.4         1,240         0.82       good
1    SB-01  2024-02-15          13.1         1,180         0.79       good
2    SB-01  2024-03-15          15.8         2,050         1.21       good
3    SB-01  2024-04-15          18.2         1,620         0.94       good
4    sb-01  2024-05-15          21.5           980         0.66       good
5   SB-01   2024-06-15        -999.0           540         0.51    suspect
6    SB-02  2024-01-15          11.8         3,410         1.88       good
7    SB-02  2024-02-15          12.3         3,290         1.92       good
8    SB-02  2024-03-15          14.9         4,870         2.41       good
9    SB-02  2024-04-15          17.6         3,950         2.09  estimated
10   SB-02  2024-05-15          20.9         2,240         1.44       good
11   sb-02  2024-06-15        -999.0         1,130         1.02    suspect
12   SB-03  2024-01-15   

1. How many monitoring sites are in the data?
   - 3
2. What is a plausible range for stream water temperature in degrees Celsius?
   - 10-22 degrees C
3. Roughly what mean water temperature do you expect across all measurements?
   - 16 degrees C

In [3]:
stream['discharge_cfs'] * 0.0283168

TypeError: can't multiply sequence by non-int of type 'float'

In [4]:
print(stream.dtypes)

site_id           object
date              object
water_temp_c     float64
discharge_cfs     object
nitrate_mg_l      object
qc_flag           object
dtype: object


In [5]:
stream['discharge_cfs'].iloc[0] * 0.0283168  # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

TypeError: can't multiply sequence by non-int of type 'float'

In [6]:
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [7]:
discharge_cms = (stream['discharge_cfs']
                .str.replace(',', '', regex=False)
                .astype(float)) * 0.0283168

discharge_cms.head(3)

0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [8]:
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

In [9]:
stream['site_id'].strip()

AttributeError: 'Series' object has no attribute 'strip'

In [10]:
stream['site_id'] = stream['site_id'].str.strip().str.lower()
stream['site_id'].unique()

array(['sb-01', 'sb-02', 'sb-03'], dtype=object)

In [11]:
stream['nitrate_mg_l'].astype(float)

ValueError: could not convert string to float: '<0.05'

In [ ]:
stream['nitrate_mg_l'].unique()

Deciding to replace <0.05 with 0.05. We figured overestimating is better than underestimating nitrate concentrations

In [12]:
stream['nitrate_mg_l'] = (stream['nitrate_mg_l']
                         .str.replace('<', '', regex=False)
                         .astype(float))
stream['nitrate_mg_l']

0     0.82
1     0.79
2     1.21
3     0.94
4     0.66
5     0.51
6     1.88
7     1.92
8     2.41
9     2.09
10    1.44
11    1.02
12    0.05
13    0.05
14    0.33
15    0.28
16    0.05
17    0.05
Name: nitrate_mg_l, dtype: float64

In [13]:
stream['temperature'].mean()

KeyError: 'temperature'

In [18]:
stream['water_temp_c'].replace(-999.0, np.nan).mean()

15.226666666666668

In [19]:
stream.iloc[18]

IndexError: single positional indexer is out-of-bounds

In [32]:
len(stream)

18

In [31]:
stream.iloc[len(stream)-1]

site_id               sb-03
date             2024-06-15
water_temp_c         -999.0
discharge_cfs           180
nitrate_mg_l           0.05
qc_flag             suspect
Name: 17, dtype: object

**KeyError:** variable is undefined
**AttributeError:** 
**TypeError:**	
**ValueError:**	Issue with unexpected value
**IndexError:** Index is out of bounds

In [37]:
stream['water_temp_c'] = stream['water_temp_c'].replace(-999.0, np.nan)
stream['water_temp_c'].mean()

15.226666666666668

In [38]:
stream['site_id'].nunique()

3

In [39]:
site_means = stream.groupby('site_id')['water_temp_c'].mean()
stream['site_mean'] = site_means
stream[['site_id', 'water_temp_c', 'site_mean']].head()

,site_id,water_temp_c,site_mean
0,sb-01,12.4,NaN
1,sb-01,13.1,NaN
2,sb-01,15.8,NaN
3,sb-01,18.2,NaN
4,sb-01,21.5,NaN


In [40]:
site_means.index

Index(['sb-01', 'sb-02', 'sb-03'], dtype='object', name='site_id')

In [41]:
site_means

site_id
sb-01    16.20
sb-02    15.50
sb-03    13.98
Name: water_temp_c, dtype: float64

In [42]:
stream.groupby('site_id')['water_temp_c'].mean()

site_id
sb-01    16.20
sb-02    15.50
sb-03    13.98
Name: water_temp_c, dtype: float64